<a href="https://colab.research.google.com/github/Valentine250/Easyvisa-Project/blob/main/flask_api_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install flask yfinance pyngrok pandas numpy --quiet

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


## STEP 2 — Import Libraries

Now that the toolboxes are installed, we 'open' them by importing them into our Python script.

In [2]:
# Flask
from flask import Flask, request, jsonify

# yfinance talks to Yahoo Finance to get stock data
import yfinance as yf

# pandas and numpy
import pandas as pd
import numpy as np

import threading
import json

# pyngrok creates a public internet URL for our local server
from pyngrok import ngrok

print("✅ All libraries imported!")

✅ All libraries imported!


In [3]:
# Create the Flask application object
# __name__ tells Flask where to find resources
app = Flask(__name__)

print("✅ Flask app created!")

✅ Flask app created!


In [4]:
@app.route('/company/<symbol>', methods=['GET'])
def company_info(symbol):
    """
    ENDPOINT 1: Company Information
    Accepts: stock symbol in the URL (e.g. /company/AAPL)
    Returns: company name, industry, sector, summary, key officers
    """
    try:
        # Step 1: Convert symbol to uppercase to be safe (aapl -> AAPL)
        symbol = symbol.upper()

        # Step 2: Ask Yahoo Finance for this company's data
        # yf.Ticker is like pointing at a company on the stock exchange
        ticker = yf.Ticker(symbol)
        info = ticker.info  # This is a big dictionary of company details

        # Step 3: Pull out only the fields we care about
        company_data = {
            "symbol": symbol,
            "full_name": info.get("longName", "N/A"),           # Full company name
            "short_name": info.get("shortName", "N/A"),         # Short name
            "industry": info.get("industry", "N/A"),             # e.g. Consumer Electronics
            "sector": info.get("sector", "N/A"),                 # e.g. Technology
            "country": info.get("country", "N/A"),               # Where they're based
            "website": info.get("website", "N/A"),               # Official website
            "employees": info.get("fullTimeEmployees", "N/A"),   # Number of employees
            "business_summary": info.get("longBusinessSummary", "N/A"),  # Description
            "key_officers": [
                # Loop through company officers and extract their name and title
                {"name": officer.get("name", "N/A"), "title": officer.get("title", "N/A")}
                for officer in info.get("companyOfficers", [])[:5]  # First 5 officers only
            ]
        }

        # Step 4: Return data as JSON (like a structured text response)
        return jsonify({"status": "success", "data": company_data}), 200

    except Exception as e:
        # If something goes wrong, return a helpful error message
        return jsonify({"status": "error", "message": str(e)}), 400

print("✅ Endpoint 1 (Company Info) registered!")

✅ Endpoint 1 (Company Info) registered!


In [5]:
@app.route('/stock/<symbol>', methods=['GET'])
def stock_market_data(symbol):
    """
    ENDPOINT 2: Real-Time Stock Market Data
    Accepts: stock symbol in the URL (e.g. /stock/AAPL)
    Returns: current price, price change, % change, market state, volume, etc.
    """
    try:
        symbol = symbol.upper()
        ticker = yf.Ticker(symbol)
        info = ticker.info

        # Calculate price change = current price minus previous close
        current_price = info.get("currentPrice") or info.get("regularMarketPrice", 0)
        previous_close = info.get("previousClose", 0)

        # Price change (how much did it move today in dollars?)
        price_change = round(current_price - previous_close, 4)

        # Percentage change (how much did it move today in percent?)
        pct_change = round((price_change / previous_close) * 100, 4) if previous_close else 0

        stock_data = {
            "symbol": symbol,
            "company_name": info.get("shortName", "N/A"),
            "market_state": info.get("marketState", "N/A"),       # PRE / REGULAR / POST / CLOSED
            "currency": info.get("currency", "USD"),
            "current_price": current_price,
            "previous_close": previous_close,
            "price_change": price_change,                          # e.g. +1.23
            "percent_change": f"{pct_change}%",                   # e.g. +0.62%
            "open_price": info.get("open", "N/A"),                # Price at market open
            "day_high": info.get("dayHigh", "N/A"),               # Highest price today
            "day_low": info.get("dayLow", "N/A"),                 # Lowest price today
            "52_week_high": info.get("fiftyTwoWeekHigh", "N/A"),  # 52-week high
            "52_week_low": info.get("fiftyTwoWeekLow", "N/A"),    # 52-week low
            "volume": info.get("volume", "N/A"),                  # Shares traded today
            "avg_volume": info.get("averageVolume", "N/A"),       # Average daily volume
            "market_cap": info.get("marketCap", "N/A"),           # Total company value
            "pe_ratio": info.get("trailingPE", "N/A"),            # Price-to-earnings ratio
        }

        return jsonify({"status": "success", "data": stock_data}), 200

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

print("✅ Endpoint 2 (Stock Market Data) registered!")

✅ Endpoint 2 (Stock Market Data) registered!


In [6]:
@app.route('/historical', methods=['POST'])
def historical_data():
    """
    ENDPOINT 3: Historical Market Data
    Accepts: JSON body with symbol, start_date, end_date
    Example body: {"symbol": "AAPL", "start_date": "2024-01-01", "end_date": "2024-12-31"}
    Returns: daily open/high/low/close/volume data for that date range
    """
    try:
        # Step 1: Read the JSON data the user sent in the request body
        body = request.get_json()

        # Step 2: Validate — make sure all required fields are present
        if not body:
            return jsonify({"status": "error", "message": "No JSON body provided"}), 400

        symbol = body.get("symbol", "").upper()
        start_date = body.get("start_date")   # Format: YYYY-MM-DD
        end_date = body.get("end_date")       # Format: YYYY-MM-DD

        if not all([symbol, start_date, end_date]):
            return jsonify({
                "status": "error",
                "message": "Missing fields. Required: symbol, start_date, end_date"
            }), 400

        # Step 3: Ask Yahoo Finance for historical price data
        ticker = yf.Ticker(symbol)
        hist_df = ticker.history(start=start_date, end=end_date)  # Returns a DataFrame (table)

        if hist_df.empty:
            return jsonify({"status": "error", "message": f"No data found for {symbol} in this range"}), 404

        # Step 4: Convert the DataFrame to a list of daily records
        records = []
        for date, row in hist_df.iterrows():
            records.append({
                "date": str(date.date()),          # The trading date
                "open": round(row["Open"], 4),     # Opening price
                "high": round(row["High"], 4),     # Highest price of the day
                "low": round(row["Low"], 4),       # Lowest price of the day
                "close": round(row["Close"], 4),   # Closing price
                "volume": int(row["Volume"]),       # Number of shares traded
            })

        return jsonify({
            "status": "success",
            "symbol": symbol,
            "start_date": start_date,
            "end_date": end_date,
            "total_trading_days": len(records),
            "data": records
        }), 200

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

print("✅ Endpoint 3 (Historical Data) registered!")

✅ Endpoint 3 (Historical Data) registered!


In [7]:
@app.route('/insights', methods=['POST'])
def analytical_insights():
    """
    ENDPOINT 4: Analytical Insights
    Accepts: JSON body with symbol, start_date, end_date
    Returns: statistical analysis, moving averages, performance metrics, signal
    """
    try:
        body = request.get_json()
        if not body:
            return jsonify({"status": "error", "message": "No JSON body provided"}), 400

        symbol = body.get("symbol", "").upper()
        start_date = body.get("start_date")
        end_date = body.get("end_date")

        if not all([symbol, start_date, end_date]):
            return jsonify({
                "status": "error",
                "message": "Missing fields. Required: symbol, start_date, end_date"
            }), 400

        # --- Fetch the historical data ---
        ticker = yf.Ticker(symbol)
        hist_df = ticker.history(start=start_date, end=end_date)

        if hist_df.empty:
            return jsonify({"status": "error", "message": f"No data found for {symbol}"}), 404

        close_prices = hist_df["Close"]  # We'll analyze closing prices

        # --- ANALYSIS 1: Price Performance ---
        start_price = round(float(close_prices.iloc[0]), 4)    # First day price
        end_price = round(float(close_prices.iloc[-1]), 4)     # Last day price
        total_return_pct = round(((end_price - start_price) / start_price) * 100, 2)
        performance_label = "📈 Positive" if total_return_pct > 0 else "📉 Negative"

        # --- ANALYSIS 2: Moving Averages ---
        # A moving average smooths out daily noise to show the trend
        # 20-day MA = average of last 20 trading days
        ma20 = round(float(close_prices.rolling(window=20).mean().iloc[-1]), 4) if len(close_prices) >= 20 else None
        ma50 = round(float(close_prices.rolling(window=50).mean().iloc[-1]), 4) if len(close_prices) >= 50 else None

        # Golden Cross Signal: if short MA > long MA → bullish (BUY signal)
        # Death Cross Signal: if short MA < long MA → bearish (SELL signal)
        if ma20 and ma50:
            signal = "🟢 BUY Signal (MA20 > MA50 — Bullish Trend)" if ma20 > ma50 else "🔴 SELL Signal (MA20 < MA50 — Bearish Trend)"
        else:
            signal = "⚠️ Not enough data for moving average signal"

        # --- ANALYSIS 3: Volatility ---
        # Daily returns = how much % the price changed each day
        daily_returns = close_prices.pct_change().dropna()
        volatility = round(float(daily_returns.std() * np.sqrt(252)) * 100, 2)  # Annualized volatility
        if volatility < 15:
            volatility_label = "🟢 Low Volatility (Stable)"
        elif volatility < 30:
            volatility_label = "🟡 Medium Volatility"
        else:
            volatility_label = "🔴 High Volatility (Risky)"

        # --- ANALYSIS 4: Best and Worst Days ---
        daily_returns_with_dates = hist_df["Close"].pct_change().dropna()
        best_day_date = str(daily_returns_with_dates.idxmax().date())
        best_day_pct = round(float(daily_returns_with_dates.max()) * 100, 2)
        worst_day_date = str(daily_returns_with_dates.idxmin().date())
        worst_day_pct = round(float(daily_returns_with_dates.min()) * 100, 2)

        # --- ANALYSIS 5: Price Statistics ---
        stats = {
            "average_close": round(float(close_prices.mean()), 4),
            "median_close": round(float(close_prices.median()), 4),
            "highest_close": round(float(close_prices.max()), 4),
            "lowest_close": round(float(close_prices.min()), 4),
        }

        # --- Package all insights ---
        insights = {
            "symbol": symbol,
            "period": {"from": start_date, "to": end_date},
            "trading_days_analyzed": len(close_prices),
            "performance": {
                "start_price": start_price,
                "end_price": end_price,
                "total_return_percent": f"{total_return_pct}%",
                "assessment": performance_label
            },
            "moving_averages": {
                "ma_20_day": ma20,
                "ma_50_day": ma50,
                "signal": signal
            },
            "volatility": {
                "annualized_volatility_percent": f"{volatility}%",
                "assessment": volatility_label
            },
            "notable_days": {
                "best_day": {"date": best_day_date, "return": f"+{best_day_pct}%"},
                "worst_day": {"date": worst_day_date, "return": f"{worst_day_pct}%"}
            },
            "price_statistics": stats
        }

        return jsonify({"status": "success", "insights": insights}), 200

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

print("✅ Endpoint 4 (Analytical Insights) registered!")

✅ Endpoint 4 (Analytical Insights) registered!


In [10]:


import os
import time

# Start Flask in a background thread so the notebook stays interactive
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# Set up ngrok tunnel
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

try:
    public_url = ngrok.connect(5000)
    BASE_URL = str(public_url)
    print(f"\n🌐 Your public API URL is: {BASE_URL}")
    print(f"\n📌 Try these URLs in your browser or Postman:")
    print(f"   {BASE_URL}/company/AAPL")
    print(f"   {BASE_URL}/stock/AAPL")
except Exception:
    BASE_URL = "http://127.0.0.1:5000"
    print(f"\n⚠️  ngrok not available — using local URL: {BASE_URL}")
    print("   (You can still test with the cells below — they run inside this notebook!)")

print("\n✅ Flask server is running!")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


ERROR:pyngrok.process.ngrok:t=2026-06-10T14:26:55+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-10T14:26:55+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-10T14:26:55+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut


⚠️  ngrok not available — using local URL: http://127.0.0.1:5000
   (You can still test with the cells below — they run inside this notebook!)

✅ Flask server is running!


In [8]:
import requests

# Helper function to print API responses nicely
def pretty_print(title, response):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"  HTTP Status: {response.status_code} ({'OK' if response.status_code == 200 else 'ERROR'})")
    print(f"{'='*60}")
    data = response.json()
    print(json.dumps(data, indent=2))

# We'll use Apple (AAPL) as our example company
SYMBOL = "AAPL"
print(f"Ready to test with symbol: {SYMBOL}")

Ready to test with symbol: AAPL


In [11]:
# ============================================================
# TEST ENDPOINT 1: Company Information
# ============================================================
print("📡 Calling: GET /company/AAPL")
response = requests.get(f"{BASE_URL}/company/{SYMBOL}")
pretty_print("ENDPOINT 1 — Company Information", response)

📡 Calling: GET /company/AAPL


INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 14:27:02] "GET /company/AAPL HTTP/1.1" 200 -



  ENDPOINT 1 — Company Information
  HTTP Status: 200 (OK)
{
  "data": {
    "business_summary": "Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download applications and digital content, such as books, music, video, games, and podcasts, as well as advertising services include third-party licensing arrangements and its own advertising platforms. In addition, the company offers various subscription-based services, such as Apple Arcade, a game subscription service;

In [12]:
# ============================================================
# TEST ENDPOINT 2: Real-Time Stock Data
# ============================================================
print("📡 Calling: GET /stock/AAPL")
response = requests.get(f"{BASE_URL}/stock/{SYMBOL}")
pretty_print("ENDPOINT 2 — Real-Time Stock Market Data", response)

📡 Calling: GET /stock/AAPL


INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 14:27:11] "GET /stock/AAPL HTTP/1.1" 200 -



  ENDPOINT 2 — Real-Time Stock Market Data
  HTTP Status: 200 (OK)
{
  "data": {
    "52_week_high": 317.4,
    "52_week_low": 195.07,
    "avg_volume": 45829400,
    "company_name": "Apple Inc.",
    "currency": "USD",
    "current_price": 289.85,
    "day_high": 291.18,
    "day_low": 287.38,
    "market_cap": 4257130217472,
    "market_state": "REGULAR",
    "open_price": 290.765,
    "pe_ratio": 35.133335,
    "percent_change": "-0.2409%",
    "previous_close": 290.55,
    "price_change": -0.7,
    "symbol": "AAPL",
    "volume": 10277353
  },
  "status": "success"
}


In [13]:
# ============================================================
# TEST ENDPOINT 3: Historical Market Data
# We send a JSON body specifying what data we want
# ============================================================
print("📡 Calling: POST /historical")

payload = {
    "symbol": SYMBOL,
    "start_date": "2024-01-01",
    "end_date": "2024-03-31"   # First quarter of 2024
}

response = requests.post(f"{BASE_URL}/historical", json=payload)
pretty_print("ENDPOINT 3 — Historical Market Data (first 5 days shown)", response)

# Show only the first 5 records to keep output manageable
data = response.json()
if data.get("status") == "success":
    print(f"\n📊 Total trading days in range: {data['total_trading_days']}")
    print("\nFirst 5 days of data:")
    for day in data["data"][:5]:
        print(f"  {day['date']} | Open: ${day['open']} | Close: ${day['close']} | Volume: {day['volume']:,}")

📡 Calling: POST /historical


INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 14:27:15] "POST /historical HTTP/1.1" 200 -



  ENDPOINT 3 — Historical Market Data (first 5 days shown)
  HTTP Status: 200 (OK)
{
  "data": [
    {
      "close": 183.5622,
      "date": "2024-01-02",
      "high": 186.3308,
      "low": 181.8318,
      "open": 185.0553,
      "volume": 82488700
    },
    {
      "close": 182.1877,
      "date": "2024-01-03",
      "high": 183.7995,
      "low": 181.3769,
      "open": 182.1581,
      "volume": 58414500
    },
    {
      "close": 179.874,
      "date": "2024-01-04",
      "high": 181.0408,
      "low": 178.8555,
      "open": 180.1113,
      "volume": 71983600
    },
    {
      "close": 179.1521,
      "date": "2024-01-05",
      "high": 180.7144,
      "low": 178.1534,
      "open": 179.953,
      "volume": 62379700
    },
    {
      "close": 183.4831,
      "date": "2024-01-08",
      "high": 183.5226,
      "low": 179.4685,
      "open": 180.0519,
      "volume": 59144500
    },
    {
      "close": 183.0678,
      "date": "2024-01-09",
      "high": 183.0777,
      "low"

In [14]:
# ============================================================
# TEST ENDPOINT 4: Analytical Insights
# ============================================================
print("📡 Calling: POST /insights")

payload = {
    "symbol": SYMBOL,
    "start_date": "2024-01-01",
    "end_date": "2024-12-31"   # Full year of 2024
}

response = requests.post(f"{BASE_URL}/insights", json=payload)
pretty_print("ENDPOINT 4 — Analytical Insights", response)

INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 14:27:24] "POST /insights HTTP/1.1" 200 -


📡 Calling: POST /insights

  ENDPOINT 4 — Analytical Insights
  HTTP Status: 200 (OK)
{
  "insights": {
    "moving_averages": {
      "ma_20_day": 247.6855,
      "ma_50_day": 235.713,
      "signal": "\ud83d\udfe2 BUY Signal (MA20 > MA50 \u2014 Bullish Trend)"
    },
    "notable_days": {
      "best_day": {
        "date": "2024-06-11",
        "return": "+7.26%"
      },
      "worst_day": {
        "date": "2024-08-05",
        "return": "-4.82%"
      }
    },
    "performance": {
      "assessment": "\ud83d\udcc8 Positive",
      "end_price": 250.5989,
      "start_price": 183.5622,
      "total_return_percent": "36.52%"
    },
    "period": {
      "from": "2024-01-01",
      "to": "2024-12-31"
    },
    "price_statistics": {
      "average_close": 205.2846,
      "highest_close": 257.3756,
      "lowest_close": 163.3614,
      "median_close": 212.4001
    },
    "symbol": "AAPL",
    "trading_days_analyzed": 251,
    "volatility": {
      "annualized_volatility_percent": "22.